# نوت‌بوک ۱ — Baseline: تنظیم دقیق ParsBERT

**هدف:** آموزش طبقه‌بند ParsBERT روی کل دیتاست و گزارش عملکرد پایه روی test set.

**ورودی:** پیکره با ۲۸۰۰ نمونه آموزش، ۴۰۰ اعتبارسنجی، ۸۰۰ آزمون.

**خروجی:** 
- `baseline_model/` — مدل آموزش‌دیده (روی Google Drive)
- `results/baseline_results.json` — متریک‌ها
- `results/baseline_predictions.jsonl` — پیش‌بینی‌ها به تفکیک نمونه (برای تحلیل نوت‌بوک ۲)

**زمان تخمینی:** ۲۰–۳۰ دقیقه روی T4.

**مقاومت در برابر قطع:** checkpoint بعد از هر epoch روی Google Drive ذخیره می‌شود. اگر قطع شدید، فقط دوباره از سلول اول اجرا کنید؛ از آخرین checkpoint ادامه می‌دهد.

## گام ۱ — اتصال Google Drive و آپلود داده

اولین بار که این نوت‌بوک را اجرا می‌کنید:
1. فولدر `persian-ai-text-corpus/data/splits/` را روی Google Drive خود (در `MyDrive/persian-ai-research/data/splits/`) آپلود کنید.
2. سه فایل لازم: `train.jsonl`, `val.jsonl`, `test.jsonl`.

سلول زیر Drive را متصل می‌کند و فایل‌ها را چک می‌کند.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# مسیرهای اصلی روی Drive — همه چیز اینجا ذخیره می‌شود
DRIVE_ROOT = Path('/content/drive/MyDrive/persian-ai-research')
DATA_DIR = DRIVE_ROOT / 'data' / 'splits'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints' / 'baseline'
RESULTS_DIR = DRIVE_ROOT / 'results'
MODEL_DIR = DRIVE_ROOT / 'models' / 'baseline_model'

for d in [CHECKPOINT_DIR, RESULTS_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# تأیید وجود داده
for fname in ['train.jsonl', 'val.jsonl', 'test.jsonl']:
    p = DATA_DIR / fname
    if not p.exists():
        raise FileNotFoundError(f'{p} یافت نشد. لطفاً ابتدا داده را روی Drive آپلود کنید.')
    print(f'✓ {fname} ({p.stat().st_size / 1024:.0f} KB)')

print(f'\nهمه مسیرها آماده. ریشه: {DRIVE_ROOT}')

## گام ۲ — نصب کتابخانه‌ها

Colab از قبل بیشترشان را دارد، فقط مطمئن می‌شویم نسخه‌ها سازگارند.

In [ ]:
!pip install -q 'transformers>=4.40,<5' 'datasets>=2.18' 'accelerate>=0.30' 'scikit-learn>=1.4' 'hazm>=0.9' 2>&1 | tail -3

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'حافظه: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## گام ۳ — بارگذاری داده

از فرمت JSONL استاندارد ما می‌خوانیم. فقط دو فیلد لازم داریم: `text` و `label`.

In [ ]:
import json
from datasets import Dataset, DatasetDict

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

raw = {
    'train': load_jsonl(DATA_DIR / 'train.jsonl'),
    'validation': load_jsonl(DATA_DIR / 'val.jsonl'),
    'test': load_jsonl(DATA_DIR / 'test.jsonl'),
}

# نگه‌داشتن متاداده (برای تحلیل به تفکیک مدل و حوزه در نوت‌بوک ۲)
ds = DatasetDict({
    split: Dataset.from_list([
        {
            'text': r['text'],
            'label': r['label'],
            'model': r.get('model', 'human'),
            'category': r.get('category', 'unknown'),
            'human_id': r.get('human_id', ''),
            'id': r.get('id', ''),
        }
        for r in records
    ])
    for split, records in raw.items()
})

for split in ds:
    n_human = sum(1 for r in ds[split] if r['label'] == 0)
    n_ai = sum(1 for r in ds[split] if r['label'] == 1)
    print(f"{split:11s}: {len(ds[split]):4d} ({n_human} انسان / {n_ai} AI)")

## گام ۴ — توکنایزر ParsBERT

از مدل `HooshvareLab/bert-fa-base-uncased` استفاده می‌کنیم — رایج‌ترین مدل پایه فارسی روی Hugging Face.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = 'HooshvareLab/bert-fa-base-uncased'
MAX_LENGTH = 256  # طول نمونه‌ها معمولاً ۴۰–۲۵۰ کلمه است؛ ۲۵۶ توکن کافی است

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch['text'],
        padding=False,            # padding داینامیک توسط collator انجام می‌شود
        truncation=True,
        max_length=MAX_LENGTH,
    )

ds_tok = ds.map(tokenize_batch, batched=True, remove_columns=['text'])
print('توکنایز کامل شد.')
print(f'نمونه‌ای از کلیدها: {list(ds_tok["train"][0].keys())}')

## گام ۵ — تنظیم مدل و پارامترهای آموزش

**انتخاب‌های کلیدی و دلیلشان:**
- `num_train_epochs=3`: کافی برای fine-tuning روی ۲۸۰۰ نمونه. بیشتر = overfitting.
- `batch_size=16`: روی T4 با حافظه ۱۵ گیگ راحت جا می‌شود.
- `learning_rate=2e-5`: استاندارد BERT fine-tuning.
- `class_weight`: کلاس انسان (label=0) سه برابر وزن می‌گیرد چون نسبت ۱:۳ است.
- `save_strategy='epoch'`: بعد از هر epoch checkpoint روی Drive ذخیره می‌شود.
- `fp16=True`: نصف حافظه + ~۱.۵ برابر سریع‌تر.

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# مدل با ۲ کلاس (انسان / AI)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'human', 1: 'ai'},
    label2id={'human': 0, 'ai': 1},
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro'),
        'f1_human': f1_score(labels, preds, pos_label=0),
        'f1_ai': f1_score(labels, preds, pos_label=1),
        'precision_macro': precision_score(labels, preds, average='macro'),
        'recall_macro': recall_score(labels, preds, average='macro'),
    }

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,           # فقط ۲ checkpoint آخر — صرفه‌جویی فضای Drive
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    logging_steps=50,
    report_to='none',             # غیرفعال‌سازی wandb و غیره
    seed=42,
)

# Trainer سفارشی با class weighting
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        # کلاس ۰ (انسان) سه برابر وزن می‌گیرد چون ۱:۳ کم است
        weights = torch.tensor([3.0, 1.0], device=logits.device)
        loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=ds_tok['train'],
    eval_dataset=ds_tok['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print('Trainer آماده. شروع آموزش...')

## گام ۶ — آموزش (با ادامه از checkpoint قبلی)

**اگر قطع شدید:** کافی است از سلول اول دوباره اجرا کنید. این سلول خودکار آخرین checkpoint را پیدا می‌کند و ادامه می‌دهد.

In [ ]:
import os

# جستجو برای آخرین checkpoint
existing_ckpts = sorted(
    [d for d in CHECKPOINT_DIR.glob('checkpoint-*') if d.is_dir()],
    key=lambda p: int(p.name.split('-')[1])
)
resume_from = str(existing_ckpts[-1]) if existing_ckpts else None

if resume_from:
    print(f'ادامه از: {resume_from}')
else:
    print('شروع از صفر (هیچ checkpoint قبلی یافت نشد).')

train_result = trainer.train(resume_from_checkpoint=resume_from)

print(f'\n✓ آموزش تمام شد.')
print(f'Loss نهایی: {train_result.training_loss:.4f}')

## گام ۷ — ارزیابی روی test set

این عددها متریک پایه مقاله هستند.

In [ ]:
import json
from datetime import datetime

test_results = trainer.evaluate(ds_tok['test'])

print('=== نتایج روی test set ===')
for k, v in test_results.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')

# ذخیره نتایج به‌صورت ساختاریافته
results_summary = {
    'experiment': 'baseline',
    'model': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'train_size': len(ds_tok['train']),
    'test_size': len(ds_tok['test']),
    'train_loss': train_result.training_loss,
    'test_metrics': {k: v for k, v in test_results.items() if isinstance(v, (int, float))},
    'timestamp': datetime.now().isoformat(),
}

results_path = RESULTS_DIR / 'baseline_results.json'
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(results_summary, f, ensure_ascii=False, indent=2)
print(f'\n✓ نتایج ذخیره شد: {results_path}')

## گام ۸ — ذخیره پیش‌بینی‌ها (برای تحلیل نوت‌بوک ۲)

این فایل برای تحلیل خطا به تفکیک مدل و حوزه ضروری است.

In [ ]:
predictions = trainer.predict(ds_tok['test'])
pred_labels = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

# ذخیره با متاداده کامل
preds_path = RESULTS_DIR / 'baseline_predictions.jsonl'
with open(preds_path, 'w', encoding='utf-8') as f:
    for i, sample in enumerate(ds['test']):
        record = {
            'id': sample['id'],
            'human_id': sample['human_id'],
            'true_label': int(true_labels[i]),
            'pred_label': int(pred_labels[i]),
            'correct': bool(pred_labels[i] == true_labels[i]),
            'model': sample['model'],
            'category': sample['category'],
            'logits': predictions.predictions[i].tolist(),
        }
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

print(f'✓ {len(true_labels)} پیش‌بینی ذخیره شد: {preds_path}')

## گام ۹ — ذخیره مدل نهایی

این مدل در نوت‌بوک ۵ (آزمایش مقاومت در برابر بازنویسی) دوباره استفاده می‌شود.

In [ ]:
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))
print(f'✓ مدل نهایی ذخیره شد: {MODEL_DIR}')
print(f'   حجم: {sum(f.stat().st_size for f in MODEL_DIR.rglob("*")) / 1e6:.0f} MB')

## خلاصه و گام بعدی

**اگر اینجا رسیدید:**
- مدل پایه آموزش دیده و روی Drive ذخیره شد.
- نتایج اولیه روی test set گرفتید.
- پیش‌بینی‌ها برای تحلیل ذخیره شدند.

**عدد F1 macro روی test معیار ماست.** اگر:
- بالای ۹۰٪: عالی، تشخیص آسان است.
- ۸۰–۹۰٪: نرمال، مقاله جالبی می‌شود.
- ۷۰–۸۰٪: چالش جدی، یافته جذاب.
- زیر ۷۰٪: مدل ضعیف یا داده مشکل دارد — قبل از ادامه با من چک کنید.

**گام بعدی:** نوت‌بوک ۲ — تحلیل خطا به تفکیک مدل و حوزه. آن یکی فقط چند دقیقه طول می‌کشد چون آموزش جدید لازم نیست.